In [ ]:
# Принципы борьбы с переобучением:
# - Регуляризация настраиваемых параметров;
# - Понижение размерности признакового пространства;
#
# Способы:
# 1. L2 и L1 регуляризаторы (чаще всего используют L2, т.к. L1 склонен занулять веса);
# 2. DropOut (метод прореживания, метод исключения);
# 3. BatchNormalization;

In [ ]:
import torch
import torchvision
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder


class RavelTransform(nn.Module):
    def forward(self, item):
        return item.ravel()


class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, output_dim)
        self.layer2 = nn.Linear(num_hidden, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


model = DigitNN(28 * 28, 32, 10)

transforms = tfs.Compose([tfs.ToImage(), tfs.Grayscale,
                          tfs.ToDtype(torch.float32, scale=True),
                          RavelTransform(),])


dataset_mnist = torchvision.datasets.MNIST('datasets/mnist', download=True, train=True, transform=transforms)

d_train, d_val = data.random_split(dataset_mnist, [0.7, 0.3])
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)
train_data_val = data.DataLoader(d_val, batch_size=32, shuffle=False)


# Добавлен параметр weight_decay 
optimizer = optim.Adam(params=model.parameters(), lr=0.01, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()
epochs = 20

model_data = torch.load('weights/mnist/model_dnn_1.tar', weights_only=True)
model.load_state_dict(model_data['model'])
transforms.load_state_dict(model_data['model'])
optimizer.load_state_dict(model_data['model'])


loss_list_val = []
loss_list = []


for _e in range(epochs):
    model.train() 
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_data, leave=False)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}')

    model.eval()
    Q_val = 0
    count_val = 0

    for x_val, y_val in train_data_val:
        with torch.no_grad():
            p = model(x_val)
            loss = loss_function(p, y_val)
            Q_val += loss.item()
            count_val += 1
    
    Q_val /= count_val

    loss_list_val.append(Q_val)
    loss_list.append(loss_mean)

    print(f' | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}')


plt.plot(loss_list)
plt.plot(loss_list_val)
plt.grid()
plt.show()

In [ ]:
# DropOut - метод выключения отдельных нейронов с вероятностью p.
# Это позволяет повысить обобщающую способность каждого нейрона.
#
# Но такой метод имеет недостаток - при обучении сеть работает с неполным кол-вом нейронов,
# но при эксплуатации - все нейроны включены. 
# Это означет, что сигнал на выходе слоя (линейная комбинация весов и входных синалов) 
# при обучении не равносилен сигналу при использовании сети.
#
# Для нейтрализации данного ограничения, сигнал на выходе слоя при обучении масштабируют множителем:
# 1/q, где q = (1-p) и p это вероятность выключение одного нейрона
# и во многих фреймворках данная функция уже встроена.
#
# DropOut может как помочь, так и ухудшить результат, а параметр p подбирается эмперически в диапазоне
# от ~0.70 до ~0.15

In [ ]:
import torch
import torchvision
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder


class RavelTransform(nn.Module):
    def forward(self, item):
        return item.ravel()


class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, output_dim)
        self.dropout_1 = nn.Dropout1d(0.3) # Используем DropOut для вектора
        self.layer2 = nn.Linear(num_hidden, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.dropout_1(x)
        x = self.layer2(x)
        return x


model = DigitNN(28 * 28, 128, 10) # Усиливаем переобучение за счет лишнего числа нейронов

transforms = tfs.Compose([tfs.ToImage(), tfs.Grayscale,
                          tfs.ToDtype(torch.float32, scale=True),
                          RavelTransform(),])


dataset_mnist = torchvision.datasets.MNIST(r'C:\datasets\mnist', download=True, train=True, transform=transforms)

d_train, d_val = data.random_split(dataset_mnist, [0.7, 0.3])
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)
train_data_val = data.DataLoader(d_val, batch_size=32, shuffle=False)


optimizer = optim.Adam(params=model.parameters(), lr=0.01, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()
epochs = 20

model_data = torch.load('model_dnn_1.tar', weights_only=True)
model.load_state_dict(model_data['model'])
transforms.load_state_dict(model_data['model'])
optimizer.load_state_dict(model_data['model'])


loss_list_val = []
loss_list = []


for _e in range(epochs):
    model.train() 
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_data, leave=False)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}')

    model.eval()
    Q_val = 0
    count_val = 0

    for x_val, y_val in train_data_val:
        with torch.no_grad():
            p = model(x_val)
            loss = loss_function(p, y_val)
            Q_val += loss.item()
            count_val += 1
    
    Q_val /= count_val

    loss_list_val.append(Q_val)
    loss_list.append(loss_mean)

    print(f' | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}')


plt.plot(loss_list)
plt.plot(loss_list_val)
plt.grid()
plt.show()

In [ ]:
# В нейросетях существует эффект: небольшие изменение весов связи может сильно влиять выходной сигнал.
# Результат: при большом кол-ве скрытых слоев, самые глубокие слои обучаются медленнее и хуже, т.к.
# выходной сигнал предыдущих слоев меняется слишком значительно.
# 
# Для нейтрализации данного эффекта и эффекта переобучения в 2015 году был предложен подход BatchNormalization.
# Сигнал на выходе слоя, в пределах одного батча, нормируется таким образом, чтобы дисперсия была единичной, а
# среднее равнялось нулю

In [ ]:
import torch
import torchvision
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder


class RavelTransform(nn.Module):
    def forward(self, item):
        return item.ravel()


class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, output_dim)
        self.bm_1 = nn.BatchNorm1d(num_hidden) # Такой слой содержит два настраиваемых параметра
        self.layer2 = nn.Linear(num_hidden, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.bm_1(x)
        x = self.layer2(x)
        return x


model = DigitNN(28 * 28, 128, 10) # Усиливаем переобучение за счет лишнего числа нейронов

transforms = tfs.Compose([tfs.ToImage(), tfs.Grayscale,
                          tfs.ToDtype(torch.float32, scale=True),
                          RavelTransform(),])


dataset_mnist = torchvision.datasets.MNIST(r'C:\datasets\mnist', download=True, train=True, transform=transforms)

d_train, d_val = data.random_split(dataset_mnist, [0.7, 0.3])
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)
train_data_val = data.DataLoader(d_val, batch_size=32, shuffle=False)


optimizer = optim.Adam(params=model.parameters(), lr=0.01, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()
epochs = 20

model_data = torch.load('model_dnn_1.tar', weights_only=True)
model.load_state_dict(model_data['model'])
transforms.load_state_dict(model_data['model'])
optimizer.load_state_dict(model_data['model'])


loss_list_val = []
loss_list = []


for _e in range(epochs):
    model.train() 
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_data, leave=False)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}')

    model.eval()
    Q_val = 0
    count_val = 0

    for x_val, y_val in train_data_val:
        with torch.no_grad():
            p = model(x_val)
            loss = loss_function(p, y_val)
            Q_val += loss.item()
            count_val += 1
    
    Q_val /= count_val

    loss_list_val.append(Q_val)
    loss_list.append(loss_mean)

    print(f' | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}')


plt.plot(loss_list)
plt.plot(loss_list_val)
plt.grid()
plt.show()

In [ ]:
# L2 -> Анализ переобучения -> Перестройка модели -> BatchNormalization -> DropOut